In [1]:
import numpy as np

In [2]:
salaries = np.array([6000, 6500, 7000, 7200, 7500, 8000, 95000])

## Task 1

أتوقع أن كلام صاحب الشركة مضلل، لأن راتب المدير (95000) أعلى بكثير من البقية ويرفع المتوسط.

In [3]:
mean = salaries.mean()
median = np.median(salaries)
print(mean, median)

19600.0 7200.0


In [4]:
q1 = np.percentile(salaries, 25)
q3 = np.percentile(salaries, 75)
iqr = q3 - q1
lower = q1 - 1.5 * iqr
upper = q3 + 1.5 * iqr
print(q1, q3, iqr, lower, upper)
print(salaries[(salaries < lower) | (salaries > upper)])

6750.0 7750.0 1000.0 5250.0 9250.0
[95000]


In [5]:
s2 = salaries[:-1]
print(s2.mean(), np.median(s2))

7033.333333333333 7100.0


In [7]:
import glob
f = glob.glob("*insuranc*")[0]
print(f)

CUsersmalshDownloadsarchive (2)insurance.csv.ipynb


## Task 2

In [9]:
print(f)
print(open(f, encoding="utf-8").read()[:300])

CUsersmalshDownloadsarchive (2)insurance.csv.ipynb
{
 "cells": [
  {
   "cell_type": "code",
   "execution_count": null,
   "id": "25c599b1",
   "metadata": {},
   "outputs": [],
   "source": []
  }
 ],
 "metadata": {
  "language_info": {
   "name": "python"
  }
 },
 "nbformat": 4,
 "nbformat_minor": 5
}



In [10]:
path = r"C:\Users\malsh\Downloads\archive (2)\insurance.csv"
data = np.genfromtxt(path, delimiter=",", skip_header=1, usecols=(0, 2, 3, 6))
# columns: 0 age · 1 bmi · 2 children · 3 charges
print(data.shape)
print(data[:3])

(1338, 4)
[[1.9000000e+01 2.7900000e+01 0.0000000e+00 1.6884924e+04]
 [1.8000000e+01 3.3770000e+01 1.0000000e+00 1.7255523e+03]
 [2.8000000e+01 3.3000000e+01 3.0000000e+00 4.4494620e+03]]


In [11]:
np.set_printoptions(suppress=True)
age = data[:, 0]
bmi = data[:, 1]
children = data[:, 2]
charges = data[:, 3]
print(data[:3])

[[   19.        27.9        0.     16884.924 ]
 [   18.        33.77       1.      1725.5523]
 [   28.        33.         3.      4449.462 ]]


In [12]:
cols = {"age": age, "bmi": bmi, "children": children, "charges": charges}
print(f"{'col':10}{'mean':>12}{'median':>12}{'std':>12}{'min':>12}{'max':>12}")
for name, c in cols.items():
    print(f"{name:10}{c.mean():12.2f}{np.median(c):12.2f}{c.std():12.2f}{c.min():12.2f}{c.max():12.2f}")

col               mean      median         std         min         max
age              39.21       39.00       14.04       18.00       64.00
bmi              30.66       30.40        6.10       15.96       53.13
children          1.09        1.00        1.21        0.00        5.00
charges       13270.42     9382.03    12105.48     1121.87    63770.43


In [13]:
print(np.isnan(data).sum())
print((children == 0).mean() * 100)

0
42.89985052316891


أكثر عمود يبتعد متوسطه عن وسيطه هو charges (المتوسط 13270 والوسيط 9382).
يعني فيه عملاء قليلين تكاليفهم عالية جدًا ترفع المتوسط، فالتوزيع ملتوي لليمين.
والـ std (12105) كبير ويقارب المتوسط، يعني التكاليف متباعدة جدًا.

##Task 3

In [15]:
def outliers_iqr(x):
    q1, q3 = np.percentile(x, 25), np.percentile(x, 75)
    iqr = q3 - q1
    lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    mask = (x < lower) | (x > upper)
    print(q1, q3, iqr, lower, upper, mask.sum(), mask.mean() * 100)

outliers_iqr(charges) 
outliers_iqr(bmi)

4740.28715 16639.912515 11899.625365 -13109.1508975 34489.350562499996 139 10.388639760837071
26.29625 34.69375 8.3975 13.7 47.290000000000006 9 0.672645739910314


In [14]:
print(np.sort(charges)[-5:])
print(np.sort(bmi)[-5:])

[55135.40209 58571.07448 60021.39897 62592.87309 63770.42801]
[48.07 49.06 50.38 52.58 53.13]


In [16]:
def z_count(x):
    z = (x - x.mean()) / x.std()
    print((np.abs(z) > 3).sum())

z_count(charges)
z_count(bmi)

7
4


In [17]:
q1, q3 = np.percentile(charges, 25), np.percentile(charges, 75)
iqr = q3 - q1
mask = (charges < q1 - 1.5 * iqr) | (charges > q3 + 1.5 * iqr)

print("مع الـ Outliers:", charges.mean())
print("بدون الـ Outliers:", charges[~mask].mean())

مع الـ Outliers: 13270.422265141257
بدون الـ Outliers: 9927.753402125938


قرار: أنا أبقي القيم العالية ولا أحذفها.
نسبتها حوالي 10% من العملاء، وهذا كثير على أن تكون كلها أخطاء.
غالبًا هي تكاليف حقيقية، والسبب المتوقع عمود smoker (مدخن)، لأن تكاليف المدخنين أعلى بكثير.
قبل ما أحذف أي شيء أفتح الملف وأشوف الأعمدة اللي ما قريتها (smoker, sex, region).

## Task4

In [18]:
def minmax(x):
    return (x - x.min()) / (x.max() - x.min())

age_n = minmax(age)
bmi_n = minmax(bmi)
charges_n = minmax(charges)

for name, c in [("age", age_n), ("bmi", bmi_n), ("charges", charges_n)]:
    print(name, c.min(), c.max())

age 0.0 1.0
bmi 0.0 1.0
charges 0.0 1.0


In [19]:
print("قبل:", age[0], bmi[0], charges[0])
print("بعد:", age_n[0], bmi_n[0], charges_n[0])

قبل: 19.0 27.9 16884.924
بعد: 0.021739130434782608 0.3212267958030669 0.25161075660777127


In [20]:
charges_z = (charges - charges.mean()) / charges.std()
print(charges_z.mean(), charges_z.std())

-8.098487832841202e-17 0.9999999999999999


In [21]:
new_age = (40 - age.min()) / (age.max() - age.min())
new_charges = (20000 - charges.min()) / (charges.max() - charges.min())
print(new_age, new_charges)

0.4782608695652174 0.30133378763783253


In [22]:
print((charges.max() - charges.min()) / (age.max() - age.min()))

1361.925089347826


لأن العميل الجديد لازم يتحول بنفس مقياس الداتا الأصلية اللي تدرّب عليها الموديل.
لو حسبنا min و max له وحده، ما يصير له مدى أصلًا، وتصير قيمته غير قابلة للمقارنة مع باقي العملاء.

بدون Normalization الموديل يرى أن charges هو الأهم لأن أرقامها بالآلاف،
بينما العمر والـ BMI أرقامهم صغيرة فيتجاهلهم تقريبًا.
هذا خطأ لأن الأهمية لا تعتمد على حجم الرقم، لذلك نوحّد المقياس.

## Task5

In [23]:
rng = np.random.default_rng(7)
brightness = 1000 + rng.normal(0, 1, 240)
for start in [30, 102, 174]:
    brightness[start:start + 5] -= 20

In [24]:
print(brightness.mean(), np.median(brightness))

998.6013699914547 999.8367390915745


In [25]:
z = (brightness - brightness.mean()) / brightness.std()
mask = z < -3
print(mask.sum())
print(np.arange(240)[mask])

15
[ 30  31  32  33  34 102 103 104 105 106 174 175 176 177 178]


In [26]:
idx = np.arange(240)[mask]
starts = idx[np.diff(idx, prepend=-10) > 1]
print(starts)

[ 30 102 174]


In [27]:
gaps_h = np.diff(starts)
print(gaps_h)
print(gaps_h / 24)

[72 72]
[3. 3.]


In [28]:
drop = np.median(brightness) - brightness[mask].mean()
print(drop, drop / np.median(brightness) * 100)

20.022076170274545 2.0025345526376714


لو طبقنا قاعدة "احذف كل الـ Outliers" هنا، كنا بنحذف القراءات المنخفضة وهي بالضبط الإشارة اللي نبحث عنها، يعني كنا بنخسر اكتشاف الكوكب.
الدرس: الـ Outlier ممكن يكون أهم معلومة في الداتا، فلازم أفهم سببه قبل ما أحذفه.